# CUT the ones that are not one KICK 

In [1]:
# ===========================================
# -----######-----######  CORE FUNCTION  ######
# ===========================================

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

def _kick_0403_i2_GET_df_onlyONEkick_mp3(
    folder_path,
    sr=44100,

    # --- core kick shape ---
    peak_must_be_within_ms=120,      # stricter than before
    min_duration_ms=70,
    max_duration_ms=1200,            # stricter (kills loops)

    # --- frequency dominance ---
    low_band_hz=(20, 160),
    high_band_hz=(220, 5000),
    min_low_to_high_ratio=3.0,       # stricter than before

    # --- decay quality ---
    min_decay_r2=0.68,               # stricter
    max_postpeak_rise_frac=0.12,     # stricter (kick mostly decays)

    # --- ONE KICK ONLY gates ---
    max_onset_peaks=1,               # must be <= 1
    onset_peak_wait_ms=140,          # minimum separation between hits
    onset_peak_rel_thresh=0.55,      # peak threshold relative to max onset
    max_env_peaks=1,                 # RMS envelope peaks must be <= 1
    env_peak_rel_thresh=0.62,        # only count strong envelope peaks
    retrigger_rms_rel=0.32,          # if RMS rises above this after decay => reject
    retrigger_min_ms=130,            # retrigger check starts after this time

    # --- file actions ---
    erase_mode="move",               # "move" (recommended) or "delete"
    quarantine_folder_name="_TRASH_NOT_KICKS",
    dry_run=True,
    audio_extensions=(".mp3",),
    verbose=False,
):
    """
    Keeps ONLY samples that look like ONE SINGLE kick hit.
    Non-kicks (including double-kicks / rolls / multiple hits) are moved/deleted immediately.
    Returns df diagnostics.
    """

    import librosa

    def _is_bad_hidden_file(fn):
        return fn.startswith("._") or fn.startswith(".DS") or fn.startswith("._DS")

    def _hz_to_bin(hz, n_fft, sr_):
        return int(np.clip(np.round(hz * n_fft / sr_), 0, n_fft // 2))

    def _count_peaks_simple(x, rel_thresh=0.6, min_dist=4):
        """
        Count local maxima above rel_thresh*max(x) with a simple distance gate.
        """
        x = np.asarray(x, dtype=float)
        if len(x) < 5:
            return 0
        mx = float(np.max(x)) + 1e-12
        thr = rel_thresh * mx

        peaks = []
        for i in range(1, len(x) - 1):
            if x[i] > thr and x[i] >= x[i-1] and x[i] >= x[i+1]:
                if not peaks or (i - peaks[-1]) >= min_dist:
                    peaks.append(i)
        return len(peaks)

    def _analyze_one(path_mp3):
        out = {
            "Path": path_mp3,
            "file_name": os.path.basename(path_mp3),
            "decision": "UNKNOWN",
            "reason": "",
            "dur_ms": np.nan,
            "peak_time_ms": np.nan,
            "low_high_ratio": np.nan,
            "decay_r2": np.nan,
            "postpeak_rise_frac": np.nan,
            "onset_peaks": np.nan,
            "env_peaks": np.nan,
            "retrigger_flag": np.nan,
            "error": "",
        }

        try:
            y, sr_ = librosa.load(path_mp3, sr=sr, mono=True)
            if y is None or len(y) < 16:
                out["decision"] = "NOT_KICK"
                out["reason"] = "empty_audio"
                return out

            dur_ms = (len(y) / sr_) * 1000.0
            out["dur_ms"] = float(dur_ms)

            if dur_ms < min_duration_ms:
                out["decision"] = "NOT_KICK"
                out["reason"] = "too_short"
                return out

            if dur_ms > max_duration_ms:
                out["decision"] = "NOT_KICK"
                out["reason"] = "too_long"
                return out

            # normalize
            peak_abs = float(np.max(np.abs(y))) + 1e-12
            y = y / peak_abs

            # absolute peak time
            pk_i = int(np.argmax(np.abs(y)))
            pk_t_ms = (pk_i / sr_) * 1000.0
            out["peak_time_ms"] = float(pk_t_ms)

            if pk_t_ms > peak_must_be_within_ms:
                out["decision"] = "NOT_KICK"
                out["reason"] = "peak_too_late"
                return out

            # start at peak (your requirement)
            y_post = y[pk_i:]
            if len(y_post) < 2048:
                out["decision"] = "NOT_KICK"
                out["reason"] = "too_short_postpeak"
                return out

            # ====== ONE-KICK Gate 1: onset peaks (multiple transients) ======
            hop = 256
            onset_env = librosa.onset.onset_strength(y=y_post, sr=sr_, hop_length=hop)
            if onset_env is None or len(onset_env) < 6:
                out["decision"] = "NOT_KICK"
                out["reason"] = "onset_failed"
                return out

            min_dist_frames = max(1, int((onset_peak_wait_ms / 1000.0) * sr_ / hop))
            onset_peaks = _count_peaks_simple(onset_env, rel_thresh=onset_peak_rel_thresh, min_dist=min_dist_frames)
            out["onset_peaks"] = int(onset_peaks)

            if onset_peaks > max_onset_peaks:
                out["decision"] = "NOT_KICK"
                out["reason"] = "multi_hit_onsets"
                return out

            # ====== RMS envelope (decay shape + envelope peaks) ======
            frame_len = 1024
            rms = librosa.feature.rms(y=y_post, frame_length=frame_len, hop_length=hop, center=False)[0]
            if rms is None or len(rms) < 8:
                out["decision"] = "NOT_KICK"
                out["reason"] = "rms_failed"
                return out

            rms = np.maximum(rms, 1e-9)
            rms = rms / (np.max(rms) + 1e-12)

            # ONE-Kick Gate 2: RMS envelope peaks must be 1
            env_peaks = _count_peaks_simple(rms, rel_thresh=env_peak_rel_thresh, min_dist=min_dist_frames)
            out["env_peaks"] = int(env_peaks)
            if env_peaks > max_env_peaks:
                out["decision"] = "NOT_KICK"
                out["reason"] = "multi_peak_envelope"
                return out

            # Gate 3: no re-trigger (after it decays, it shouldn't rise big again)
            t_ms = (np.arange(len(rms)) * hop / sr_) * 1000.0
            start_idx = int(np.argmax(t_ms >= retrigger_min_ms)) if np.any(t_ms >= retrigger_min_ms) else len(rms)
            retrigger_flag = 0
            if start_idx < len(rms):
                # Find the minimum after the initial attack, then see if it rises again significantly
                tail = rms[start_idx:]
                if len(tail) >= 4:
                    min_tail = float(np.min(tail))
                    max_tail = float(np.max(tail))
                    # if there's a strong rise later, likely a second hit
                    if (max_tail - min_tail) > 0.45 and max_tail > retrigger_rms_rel:
                        retrigger_flag = 1
            out["retrigger_flag"] = int(retrigger_flag)

            if retrigger_flag == 1:
                out["decision"] = "NOT_KICK"
                out["reason"] = "retrigger_detected"
                return out

            # post-peak rise fraction (general "wiggle" check)
            diffs = np.diff(rms)
            rises = np.sum(diffs > 0)
            out["postpeak_rise_frac"] = float(rises / max(len(diffs), 1))
            if out["postpeak_rise_frac"] > max_postpeak_rise_frac:
                out["decision"] = "NOT_KICK"
                out["reason"] = "envelope_rises_too_much"
                return out

            # decay fit on log envelope
            t = np.arange(len(rms), dtype=float)
            lr = np.log(rms)
            start = min(2, len(lr) - 1)
            t2 = t[start:]
            lr2 = lr[start:]

            t2c = t2 - np.mean(t2)
            b = float(np.sum(t2c * (lr2 - np.mean(lr2))) / (np.sum(t2c ** 2) + 1e-12))
            a = float(np.mean(lr2) - b * np.mean(t2))

            pred = a + b * t2
            ss_res = float(np.sum((lr2 - pred) ** 2))
            ss_tot = float(np.sum((lr2 - np.mean(lr2)) ** 2) + 1e-12)
            r2 = 1.0 - ss_res / ss_tot
            out["decay_r2"] = float(r2)

            if not (b < 0.0 and r2 >= min_decay_r2):
                out["decision"] = "NOT_KICK"
                out["reason"] = "decay_not_clean"
                return out

            # spectral low/high dominance (kick body)
            n_fft = 2048
            S = np.abs(librosa.stft(y_post[: min(len(y_post), sr_)], n_fft=n_fft, hop_length=hop, center=False)) ** 2
            if S is None or S.size == 0:
                out["decision"] = "NOT_KICK"
                out["reason"] = "stft_failed"
                return out

            b0 = _hz_to_bin(low_band_hz[0], n_fft, sr_)
            b1 = _hz_to_bin(low_band_hz[1], n_fft, sr_)
            h0 = _hz_to_bin(high_band_hz[0], n_fft, sr_)
            h1 = _hz_to_bin(high_band_hz[1], n_fft, sr_)

            low_energy = float(np.sum(S[b0:b1 + 1, :]))
            high_energy = float(np.sum(S[h0:h1 + 1, :])) + 1e-12
            ratio = low_energy / high_energy
            out["low_high_ratio"] = float(ratio)

            if ratio < min_low_to_high_ratio:
                out["decision"] = "NOT_KICK"
                out["reason"] = "not_lowfreq_dominant"
                return out

            out["decision"] = "KICK"
            out["reason"] = "passed_ONEKICK_rules"
            return out

        except Exception as e:
            out["decision"] = "NOT_KICK"
            out["reason"] = "exception"
            out["error"] = str(e)
            return out

    folder_path = os.path.abspath(os.path.expanduser(str(folder_path)))
    if not os.path.isdir(folder_path):
        raise ValueError(f"folder_path not found: {folder_path}")

    files = []
    exts_lower = tuple([e.lower() for e in audio_extensions])

    for fn in os.listdir(folder_path):
        if _is_bad_hidden_file(fn):
            continue
        full = os.path.join(folder_path, fn)
        if os.path.isfile(full) and fn.lower().endswith(exts_lower):
            files.append(full)

    quarantine_dir = os.path.join(folder_path, quarantine_folder_name)
    if erase_mode == "move" and (not dry_run):
        os.makedirs(quarantine_dir, exist_ok=True)

    rows = []
    for p in tqdm(files, desc="ONLY ONE-KICK — scanning mp3"):
        res = _analyze_one(p)
        rows.append(res)

        # erase right away
        if res["decision"] == "NOT_KICK" and (not dry_run):
            if erase_mode == "move":
                dest = os.path.join(quarantine_dir, os.path.basename(p))
                if os.path.exists(dest):
                    base, ext = os.path.splitext(os.path.basename(p))
                    k = 1
                    while True:
                        dest2 = os.path.join(quarantine_dir, f"{base}__dup{k}{ext}")
                        if not os.path.exists(dest2):
                            dest = dest2
                            break
                        k += 1
                shutil.move(p, dest)
            elif erase_mode == "delete":
                os.remove(p)

    df = pd.DataFrame(rows)
    df["is_kick"] = df["decision"].eq("KICK")
    df["action"] = "KEEP"
    df.loc[df["decision"].eq("NOT_KICK"), "action"] = ("DRY_RUN_SKIP" if dry_run else ("MOVE_TO_QUARANTINE" if erase_mode=="move" else "DELETE"))

    if verbose:
        n_all = len(df)
        n_k = int(df["is_kick"].sum())
        n_nk = n_all - n_k
        print("\n---- ONLY ONE-KICK SUMMARY ----")
        print(f"folder: {folder_path}")
        print(f"mp3 scanned: {n_all}")
        print(f"KEEP (KICK): {n_k}")
        print(f"ERASE (NOT_KICK): {n_nk}")
        print(f"dry_run: {dry_run} | erase_mode: {erase_mode}")
        if erase_mode == "move":
            print(f"quarantine_dir: {quarantine_dir}")

    return df

In [2]:
folder_path = r"/Users/yerik/Music/_0_YODJ_PROD/_KICKS"

df_onekick = _kick_0403_i2_GET_df_onlyONEkick_mp3(
    folder_path=folder_path,
    dry_run=True,          # <<< preview FIRST
    erase_mode="move",
    verbose=True
)

# when you're happy:
df_onekick = _kick_0403_i2_GET_df_onlyONEkick_mp3(
    folder_path=folder_path,
    dry_run=False,         # <<< actually moves/deletes
    erase_mode="move",
    verbose=True
)

ONLY ONE-KICK — scanning mp3: 100%|██████████████████████████████████████| 8944/8944 [00:17<00:00, 503.13it/s]



---- ONLY ONE-KICK SUMMARY ----
folder: /Users/yerik/Music/_0_YODJ_PROD/_KICKS
mp3 scanned: 8944
KEEP (KICK): 437
ERASE (NOT_KICK): 8507
dry_run: True | erase_mode: move
quarantine_dir: /Users/yerik/Music/_0_YODJ_PROD/_KICKS/_TRASH_NOT_KICKS


ONLY ONE-KICK — scanning mp3: 100%|██████████████████████████████████████| 8944/8944 [00:48<00:00, 183.82it/s]


---- ONLY ONE-KICK SUMMARY ----
folder: /Users/yerik/Music/_0_YODJ_PROD/_KICKS
mp3 scanned: 8944
KEEP (KICK): 437
ERASE (NOT_KICK): 8507
dry_run: False | erase_mode: move
quarantine_dir: /Users/yerik/Music/_0_YODJ_PROD/_KICKS/_TRASH_NOT_KICKS


# organize KICKS in folders 

In [3]:
# ============================================================
# 0_FNS
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa


# -----######-----######  CORE IMPORTABLE FUNCTION  #####-----######-----######
def _kick_0403_smartbucket_inplace_GET_df_manifest(
    in_dir,
    audio_extensions,
    mode="move",                 # "move" | "copy" | "none"
    sr_target=44100,
    top_db_trim=60,
    peak_db_drop=18,
    n_fft=2048,
    hop_length=256,
    min_per_bucket=12,           # prevents empty folders
    dry_run=False,               # True = no file ops
    seed=7,
):
    """
    Smart kick organizer (in-place):
    - Reads kicks from in_dir (root only)
    - Creates 9 bucket folders inside in_dir
    - Uses adaptive percentile-based + multi-check scoring per category
    - Ensures each bucket has at least min_per_bucket by rebalancing borderline samples
    - Moves (or copies) originals into those folders
    - Saves kick_manifest.csv into in_dir
    """

    rng = np.random.RandomState(seed)

    buckets = [
        "01_PUNCHY",
        "02_DEEP_SUB",
        "03_HARD_TECHNO",
        "04_SHORT_TIGHT",
        "05_LONG_BOOMY",
        "06_ANALOG",
        "07_DIRTY_DIST",
        "08_LAYER_KICKS",
        "09_WEIRD_TEXTURE",
    ]
    bucket_paths = {b: os.path.join(in_dir, b) for b in buckets}

    # ---------------- helpers ----------------
    def _safe_makedirs(p):
        os.makedirs(p, exist_ok=True)

    def _is_in_bucket_folder(path_abs):
        for b in buckets:
            b_abs = os.path.abspath(bucket_paths[b])
            if os.path.abspath(path_abs).startswith(b_abs + os.sep):
                return True
        return False

    def _list_audio_files(root, exts):
        exts_l = [e.lower() for e in exts] if exts else []
        paths = []
        for fn in os.listdir(root):
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            p = os.path.join(root, fn)
            if os.path.isfile(p):
                if _is_in_bucket_folder(p):
                    continue
                ext = os.path.splitext(fn)[1].lower()
                if (not exts_l) or (ext in exts_l):
                    paths.append(p)
        return sorted(paths)

    def _decay_ms_from_peak(x, sr, drop_db):
        env = np.abs(x)
        if env.size < 10:
            return 0.0
        win = max(16, int(0.005 * sr))  # ~5ms smoothing
        k = np.ones(win) / win
        env_s = np.convolve(env, k, mode="same")

        peak_idx = int(np.argmax(env_s))
        peak_val = float(env_s[peak_idx]) + 1e-12
        target = peak_val * (10 ** (-drop_db / 20.0))

        tail = env_s[peak_idx:]
        below = np.where(tail <= target)[0]
        if below.size == 0:
            return (len(tail) / sr) * 1000.0
        return (float(below[0]) / sr) * 1000.0

    def _band_energy_ratio(S, freqs, f_lo, f_hi):
        mask = (freqs >= f_lo) & (freqs < f_hi)
        if not np.any(mask):
            return 0.0
        num = float(np.sum(S[mask, :]))
        den = float(np.sum(S)) + 1e-12
        return num / den

    def _z(x):
        # safe z-score for a scalar, using global stats dict
        mu = stats[x]["mu"]
        sd = stats[x]["sd"]
        return lambda v: (v - mu) / sd if sd > 1e-12 else 0.0

    def _mk_dest(dst_dir, src):
        dst = os.path.join(dst_dir, os.path.basename(src))
        if not os.path.exists(dst):
            return dst
        base, ext = os.path.splitext(os.path.basename(src))
        i = 1
        while True:
            dst2 = os.path.join(dst_dir, f"{base}__DUP{i}{ext}")
            if not os.path.exists(dst2):
                return dst2
            i += 1

    def _q(series, p):
        return float(np.nanpercentile(series.to_numpy(dtype=float), p))

    # ---------------- setup folders ----------------
    for b in buckets:
        _safe_makedirs(bucket_paths[b])

    paths = _list_audio_files(in_dir, audio_extensions)

    # ---------------- feature extraction ----------------
    rows = []
    for p in tqdm(paths, desc="Extracting kick features", total=len(paths)):
        try:
            y, sr = librosa.load(p, sr=sr_target, mono=True)
            y, _ = librosa.effects.trim(y, top_db=top_db_trim)
            if y.size == 0:
                raise ValueError("empty_audio_after_trim")

            y = y / (np.max(np.abs(y)) + 1e-12)

            S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

            sub_ratio   = _band_energy_ratio(S, freqs, 20, 80)
            low_ratio   = _band_energy_ratio(S, freqs, 80, 200)
            click_ratio = _band_energy_ratio(S, freqs, 2000, 10000)
            mid_ratio   = _band_energy_ratio(S, freqs, 200, 2000)

            centroid = float(np.mean(librosa.feature.spectral_centroid(S=S, sr=sr)))
            rolloff  = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)))
            flatness = float(np.mean(librosa.feature.spectral_flatness(S=S)))
            zcr      = float(np.mean(librosa.feature.zero_crossing_rate(y)))

            decay_ms = _decay_ms_from_peak(y, sr, peak_db_drop)
            dur_ms   = (len(y) / sr) * 1000.0

            # transient proxy: stronger attack -> higher max/mean RMS ratio in early frames
            frame = hop_length
            early = y[: min(len(y), int(0.06 * sr))]  # first 60ms
            early_rms = np.sqrt(np.mean(early**2) + 1e-12)
            full_rms  = np.sqrt(np.mean(y**2) + 1e-12)
            punch = float((early_rms / (full_rms + 1e-12)))

            rows.append({
                "Path": p,
                "file_name": os.path.basename(p),
                "dur_ms": dur_ms,
                "decay_ms": decay_ms,
                "sub_ratio": sub_ratio,
                "low_ratio": low_ratio,
                "mid_ratio": mid_ratio,
                "click_ratio": click_ratio,
                "centroid_hz": centroid,
                "rolloff_hz": rolloff,
                "flatness": flatness,
                "zcr": zcr,
                "punch": punch,
                "error": ""
            })

        except Exception as e:
            rows.append({
                "Path": p,
                "file_name": os.path.basename(p),
                "dur_ms": np.nan,
                "decay_ms": np.nan,
                "sub_ratio": np.nan,
                "low_ratio": np.nan,
                "mid_ratio": np.nan,
                "click_ratio": np.nan,
                "centroid_hz": np.nan,
                "rolloff_hz": np.nan,
                "flatness": np.nan,
                "zcr": np.nan,
                "punch": np.nan,
                "error": str(e)
            })

    df = pd.DataFrame(rows)
    df_ok = df[df["error"].eq("")].copy()

    if len(df_ok) == 0:
        df.to_csv(os.path.join(in_dir, "kick_manifest.csv"), index=False)
        return df

    # ---------------- adaptive stats ----------------
    feat_cols = ["dur_ms","decay_ms","sub_ratio","low_ratio","mid_ratio","click_ratio","centroid_hz","rolloff_hz","flatness","zcr","punch"]
    stats = {}
    for c in feat_cols:
        v = df_ok[c].to_numpy(dtype=float)
        mu = float(np.nanmean(v))
        sd = float(np.nanstd(v) + 1e-12)
        stats[c] = {"mu": mu, "sd": sd}

    # percentiles (dataset-adaptive)
    Q = {
        "decay_lo": _q(df_ok["decay_ms"], 25),
        "decay_hi": _q(df_ok["decay_ms"], 75),
        "sub_hi":   _q(df_ok["sub_ratio"], 75),
        "sub_lo":   _q(df_ok["sub_ratio"], 25),
        "click_hi": _q(df_ok["click_ratio"], 75),
        "flat_hi":  _q(df_ok["flatness"], 75),
        "flat_lo":  _q(df_ok["flatness"], 25),
        "zcr_hi":   _q(df_ok["zcr"], 75),
        "punch_hi": _q(df_ok["punch"], 75),
        "dur_hi":   _q(df_ok["dur_ms"], 75),
        "dur_lo":   _q(df_ok["dur_ms"], 25),
        "cent_hi":  _q(df_ok["centroid_hz"], 75),
        "cent_lo":  _q(df_ok["centroid_hz"], 25),
    }

    # ---------------- scoring model (multi-check per category) ----------------
    # IMPORTANT: these are relative z-scores + percentile gates to avoid empty folders.
    z_decay = _z("decay_ms")
    z_sub   = _z("sub_ratio")
    z_click = _z("click_ratio")
    z_flat  = _z("flatness")
    z_zcr   = _z("zcr")
    z_cent  = _z("centroid_hz")
    z_punch = _z("punch")
    z_dur   = _z("dur_ms")

    def _scores(row):
        d = float(row["decay_ms"])
        s = float(row["sub_ratio"])
        c = float(row["click_ratio"])
        f = float(row["flatness"])
        z = float(row["zcr"])
        ce = float(row["centroid_hz"])
        p = float(row["punch"])
        du = float(row["dur_ms"])

        # gates (soft): return small bonuses, not hard if/else
        gate_deep = 0.6 if (s >= Q["sub_hi"]) else 0.0
        gate_click = 0.5 if (c >= Q["click_hi"]) else 0.0
        gate_dirty = 0.7 if (f >= Q["flat_hi"] and z >= Q["zcr_hi"]) else 0.0
        gate_short = 0.6 if (d <= Q["decay_lo"] and du <= Q["dur_lo"]) else 0.0
        gate_long  = 0.6 if (d >= Q["decay_hi"] and du >= Q["dur_hi"]) else 0.0

        # category scores (weighted combos)
        sc = {}

        # Punchy: strong attack + click, not super-long
        sc["01_PUNCHY"] = (
            1.2 * z_punch(p) +
            1.0 * z_click(c) +
            0.4 * z_cent(ce) -
            0.5 * z_decay(d)
        ) + gate_click

        # Deep Sub: lots of sub + rounder (lower centroid/click)
        sc["02_DEEP_SUB"] = (
            1.6 * z_sub(s) -
            0.6 * z_click(c) -
            0.4 * z_cent(ce) +
            0.3 * z_decay(d)
        ) + gate_deep

        # Hard Techno: brighter + denser + not too round; often some dirt
        sc["03_HARD_TECHNO"] = (
            0.9 * z_cent(ce) +
            0.7 * z_click(c) +
            0.6 * z_flat(f) -
            0.5 * z_sub(s)
        ) + (0.3 if f >= Q["flat_lo"] else 0.0)

        # Short Tight: short decay + short duration + controlled sub
        sc["04_SHORT_TIGHT"] = (
            -1.7 * z_decay(d) -
            0.9 * z_dur(du) +
            0.4 * z_punch(p) -
            0.2 * z_sub(s)
        ) + gate_short

        # Long Boomy: long decay + long duration + not necessarily huge sub
        sc["05_LONG_BOOMY"] = (
            1.4 * z_decay(d) +
            1.0 * z_dur(du) +
            0.4 * z_sub(s) -
            0.3 * z_click(c)
        ) + gate_long

        # Analog: lower flatness + rounder centroid + some sub
        sc["06_ANALOG"] = (
            -1.2 * z_flat(f) -
            0.6 * z_zcr(z) -
            0.6 * z_cent(ce) +
            0.7 * z_sub(s) +
            0.2 * z_decay(d)
        ) + (0.35 if (f <= Q["flat_lo"] and ce <= Q["cent_lo"]) else 0.0)

        # Dirty Dist: flat/noisy + high zcr
        sc["07_DIRTY_DIST"] = (
            1.8 * z_flat(f) +
            1.2 * z_zcr(z) +
            0.3 * z_cent(ce)
        ) + gate_dirty

        # Layer Kicks: both sub + click + longer body (common in “ready” kicks)
        sc["08_LAYER_KICKS"] = (
            1.0 * z_sub(s) +
            0.9 * z_click(c) +
            0.6 * z_dur(du) +
            0.2 * z_punch(p)
        ) + (0.4 if (s >= Q["sub_hi"] and c >= Q["click_hi"]) else 0.0)

        # Weird Texture: leftover catcher, but reward “unusual combos”
        # (high variance feel: very high centroid with high sub, or very low click with high flatness, etc.)
        weird_bonus = 0.0
        if (s >= Q["sub_hi"] and ce >= Q["cent_hi"]):
            weird_bonus += 0.5
        if (c <= Q["dur_lo"] and f >= Q["flat_hi"]):
            weird_bonus += 0.3
        sc["09_WEIRD_TEXTURE"] = (
            0.4 * z_flat(f) +
            0.3 * z_cent(ce) +
            0.2 * z_sub(s) +
            0.2 * z_zcr(z)
        ) + weird_bonus

        return sc

    # assign by max score, store margin (confidence)
    buckets_assigned = []
    conf_margin = []
    for r in df_ok.itertuples(index=False):
        sc = _scores(pd.Series(r._asdict()))
        items = sorted(sc.items(), key=lambda kv: kv[1], reverse=True)
        b1, s1 = items[0]
        b2, s2 = items[1]
        buckets_assigned.append(b1)
        conf_margin.append(float(s1 - s2))

    df_ok["bucket"] = buckets_assigned
    df_ok["conf_margin"] = conf_margin

    # ---------------- rebalance to prevent empty/skinny folders ----------------
    # Strategy:
    # - For buckets under min_per_bucket, steal from:
    #   - samples with LOW confidence margin (borderline)
    #   - and where that target bucket score is close to the assigned bucket score
    #
    # This preserves “obvious” kicks and redistributes only ambiguous ones.
    df_ok["_idx"] = np.arange(len(df_ok))

    # precompute all scores for speed
    all_scores = []
    for r in df_ok.itertuples(index=False):
        sc = _scores(pd.Series(r._asdict()))
        all_scores.append(sc)
    # dict list -> dataframe
    score_df = pd.DataFrame(all_scores)
    score_df["_idx"] = df_ok["_idx"].values

    def _rebalance_once(df_work):
        counts = df_work["bucket"].value_counts().to_dict()

        need = [b for b in buckets if counts.get(b, 0) < min_per_bucket]
        if not need:
            return df_work, False

        # candidates to move: low confidence first
        df_cand = df_work.sort_values("conf_margin", ascending=True).copy()

        moved_any = False
        for target in need:
            cur_n = counts.get(target, 0)
            to_add = max(0, min_per_bucket - cur_n)
            if to_add == 0:
                continue

            # choose candidates where target score is high-ish
            merged = df_cand.merge(score_df[["_idx", target]], on="_idx", how="left")
            merged = merged.rename(columns={target: "target_score"})

            # don’t move samples already in target
            merged = merged[merged["bucket"] != target].copy()

            # rank candidates:
            # 1) higher target_score
            # 2) lower conf_margin (more ambiguous)
            merged = merged.sort_values(["target_score", "conf_margin"], ascending=[False, True])

            pick = merged.head(to_add)
            if len(pick) == 0:
                continue

            # apply moves
            move_idx = pick["_idx"].to_list()
            df_work.loc[df_work["_idx"].isin(move_idx), "bucket"] = target

            moved_any = True

        return df_work, moved_any

    # rebalance a few rounds
    for _ in range(6):
        df_ok, changed = _rebalance_once(df_ok)
        if not changed:
            break

    # merge back
    df = df.merge(df_ok[["Path", "bucket", "conf_margin"]], on="Path", how="left")
    df["bucket"] = df["bucket"].fillna("09_WEIRD_TEXTURE")

    # ---------------- move/copy files in-place ----------------
    if (mode.lower() in ["move", "copy"]) and (not dry_run):
        ok_rows = df[df["error"].eq("")]
        for r in tqdm(ok_rows.itertuples(index=False), desc=f"{mode.upper()} originals into folders", total=len(ok_rows)):
            src = r.Path
            bucket = r.bucket if isinstance(r.bucket, str) else "09_WEIRD_TEXTURE"
            dst_dir = bucket_paths.get(bucket, bucket_paths["09_WEIRD_TEXTURE"])
            dst = _mk_dest(dst_dir, src)

            if os.path.abspath(src) == os.path.abspath(dst):
                continue

            if mode.lower() == "copy":
                shutil.copy2(src, dst)
            else:
                shutil.move(src, dst)

    # ---------------- save manifest + summary ----------------
    out_csv = os.path.join(in_dir, "kick_manifest.csv")
    df.to_csv(out_csv, index=False)

    # add a small summary table at bottom of df (optional convenience)
    counts = df[df["error"].eq("")]["bucket"].value_counts().reindex(buckets).fillna(0).astype(int)
    df_summary = counts.reset_index()
    df_summary.columns = ["bucket", "count"]

    # store summary in an attribute-like column (keeps it simple for you)
    # (you can ignore this; manifest.csv is the main output)
    return df, df_summary

In [4]:
# ============================================================
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
# ============================================================

audio_extensions = [".mp3"]

df_kicks, df_summary = _kick_0403_smartbucket_inplace_GET_df_manifest(
    in_dir=r"/Users/yerik/Music/_0_YODJ_PROD/_KICKS",
    audio_extensions=audio_extensions,
    mode="move",          # moves originals into the created folders
    min_per_bucket=12,    # raise/lower depending on your library
    dry_run=False
)

print(df_summary)
print(df_kicks[["file_name","bucket","conf_margin","decay_ms","sub_ratio","click_ratio","flatness","punch","error"]].head(25))

MOVE originals into folders: 100%|████████████████████████████████████████| 437/437 [00:00<00:00, 6483.77it/s]

             bucket  count
0         01_PUNCHY     22
1       02_DEEP_SUB     70
2    03_HARD_TECHNO     40
3    04_SHORT_TIGHT    147
4     05_LONG_BOOMY    108
5         06_ANALOG     17
6     07_DIRTY_DIST     11
7    08_LAYER_KICKS     12
8  09_WEIRD_TEXTURE     10
                                            file_name          bucket  \
0   _s11_b1-71412-10ABmin---25-VARIOUS-320srcbasss...  04_SHORT_TIGHT   
1   _s11_b2-71406-4AFmin---25-VARIOUS-05124basssil...  04_SHORT_TIGHT   
2   _s11_b3-7140o-5ACmin---25-VARIOUS-150srcbasssi...  04_SHORT_TIGHT   
3   _s11_b3-7141y-7ADmin---25-VARIOUS-351srcbasssi...  04_SHORT_TIGHT   
4   _s11_b3-7142c-6BA#maj---25-VARIOUS-416srcbasss...  04_SHORT_TIGHT   
5   _s11_b4-7140e-7ADmin---25-VARIOUS-099srcbasssi...   05_LONG_BOOMY   
6   _s11_b4-7141a-1AG#min---25-VARIOUS-226srcbasss...       01_PUNCHY   
7   _s11_b4-7141n-7ADmin---25-VARIOUS-311srcbasssi...  04_SHORT_TIGHT   
8   _s11_b4-7141u-8AAmin---25-VARIOUS-348srcbasssi...  03_HARD_TECHNO   
